# Fine-Tuning Ultra-Rapido con Unsloth y Exportacion a GGUF para Ollama

**Nivel:** Avanzado  
**Tecnologias:** `unsloth`, OpenAI Triton, PyTorch, GGUF, Ollama  
**Modelo Base:** Google Gemma 2 2B Instruct 4-bit (`unsloth/gemma-2-2b-it-bnb-4bit`)  
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jggomez/workshop-open-models/blob/main/session-03-fine-tuning-llms/03-fast-finetuning-unsloth-gguf/03_fast_finetuning_unsloth_gguf.ipynb)

---

## 1. Fundamentos Tecnologicos: Por que Unsloth y que es GGUF?

### La Arquitectura de Unsloth
**Unsloth** es un framework open-source de optimizacion de LLMs que reescribe los kernels computacionales de PyTorch directamente en **OpenAI Triton**:
- **Kernels Manuales:** Reemplaza multiplicaciones de atencion, RoPE (Rotary Position Embeddings), Cross-Entropy y RMSNorm por versiones compiladas a bajo nivel.
- **Backpropagation Manual:** Calcula derivadas analiticas intermedias evitando almacenar mapas de activacion masivos en VRAM.
- **Rendimiento:** 2x a 5x mayor velocidad de entrenamiento y hasta un **70% de ahorro en consumo de memoria GPU**, con exactamente 0% de perdida en precision numerica.

### El Formato Binario GGUF y el Ecosistema de Serving (Ollama / vLLM)
En entornos productivos, servir un modelo cargando checkpoints de PyTorch de 16 bits en Hugging Face es costoso e ineficiente. El formato **GGUF (Georgi Gerganov Unified Format)** es el estandar universal de `llama.cpp`:
- Almacena metadatos y tensores cuantizados en un unico archivo binario compacto.
- Permite mapeo de memoria directo (*mmap*) y offloading parcial entre CPU y GPU.
- Es el formato nativo consumido por motores de inferencia de alto rendimiento como **Ollama**, **vLLM**, **LM Studio** y **LiteRT**.

### Objetivos de este Laboratorio
1. Cargar un modelo pre-cuantizado a 4 bits con `FastLanguageModel` de Unsloth.
2. Inyectar adaptadores LoRA en todos los modulos lineales (`q`, `k`, `v`, `o`, `gate`, `up`, `down`).
3. Entrenar el modelo en una tarea de **extraccion estructurada en formato JSON estricto**.
4. Evaluar la inferencia acelerada token por token.
5. Exportar el modelo resultante directamente a formato cuantizado **GGUF** (`q4_k_m`).
6. Generar el `Modelfile` para servir el modelo inmediatamente con **Ollama** (puente hacia la Sesion 4).


### Paso 1: Instalacion Especializada de Unsloth en Google Colab (GPU CUDA)

Instalamos Unsloth y las bibliotecas complementarias de entrenamiento:


In [ ]:
# Instalacion oficial de Unsloth optimizada para Google Colab con soporte CUDA
!pip install -q --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets

### Paso 2: Carga Optimizada del Modelo Base con `FastLanguageModel`

Cargamos `unsloth/gemma-2-2b-it-bnb-4bit` con soporte de longitud de contexto de hasta 2048 tokens consumiendo menos de 4 GB de VRAM:


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # None para deteccion automatica (Float16 o Bfloat16 segun GPU)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-2-2b-it-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print("Modelo y tokenizador de Unsloth cargados exitosamente.")

### Paso 3: Inyeccion de Adaptadores LoRA en Todos los Modulos Lineales

A diferencia de SFT basico que solo entrena `q_proj` y `v_proj`, Unsloth permite entrenar de forma eficiente **todos los modulos lineales** sin impacto severo en VRAM, maximizando la capacidad de adaptacion:


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=16,
    lora_dropout=0,  # Unsloth optimiza lora_dropout=0 para maxima velocidad
    bias="none",
    use_gradient_checkpointing="unsloth",  # Ahorro masivo de memoria VRAM
    random_state=3407,
)

print("Adaptadores LoRA inyectados con kernels optimizados de Unsloth.")

### Paso 4: Preparacion de Dataset para Salida Estructurada (JSON Schema)

Entrenaremos al modelo para que actue como un clasificador corporativo que analiza tickets de soporte y devuelve **JSON estrictamente validable**:


In [ ]:
from datasets import Dataset
import json

ticket_examples = [
    {
        "instruction": "Analiza el siguiente ticket de soporte y extrae la informacion en formato JSON con las claves categoria, urgencia, sentimiento y accion_sugerida:",
        "input": "Se cayo el servidor principal de produccion en la region US-East! Todos nuestros clientes estan recibiendo error 502 y estamos perdiendo transacciones por minuto!",
        "output": json.dumps({"categoria": "infraestructura", "urgencia": "critica", "sentimiento": "frustrado", "accion_sugerida": "escalar a SRE guardia y activar cluster de contingencia"})
    },
    {
        "instruction": "Analiza el siguiente ticket de soporte y extrae la informacion en formato JSON con las claves categoria, urgencia, sentimiento y accion_sugerida:",
        "input": "Hola, me gustaria saber si para el proximo mes podrian agregar soporte para pago con tarjetas American Express en el portal web?",
        "output": json.dumps({"categoria": "facturacion", "urgencia": "baja", "sentimiento": "positivo", "accion_sugerida": "registrar feature request en backlog de pasarela de pagos"})
    },
    {
        "instruction": "Analiza el siguiente ticket de soporte y extrae la informacion en formato JSON con las claves categoria, urgencia, sentimiento y accion_sugerida:",
        "input": "Llevo dos semanas esperando la respuesta a mi ticket #4021 sobre la exportacion de auditoria SOC2. Esto es inaceptable para una cuenta Enterprise.",
        "output": json.dumps({"categoria": "cumplimiento", "urgencia": "alta", "sentimiento": "negativo", "accion_sugerida": "asignar Technical Account Manager para llamada inmediata"})
    },
    {
        "instruction": "Analiza el siguiente ticket de soporte y extrae la informacion en formato JSON con las claves categoria, urgencia, sentimiento y accion_sugerida:",
        "input": "Noto que la latencia en las consultas GraphQL aumento aproximadamente 15% tras el despliegue de anoche. No es bloqueante pero seria bueno revisar los indices.",
        "output": json.dumps({"categoria": "rendimiento", "urgencia": "media", "sentimiento": "neutral", "accion_sugerida": "revisar query plan y cache de base de datos"})
    }
]

# Formateo con plantilla de chat de Gemma
formatted_texts = []
for ex in ticket_examples:
    full_prompt = f"{ex['instruction']}\n{ex['input']}"
    messages = [
        {"role": "user", "content": full_prompt},
        {"role": "assistant", "content": ex["output"]}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    formatted_texts.append({"text": text})

dataset = Dataset.from_list(formatted_texts)
print(f"Muestras preparadas: {len(dataset)}")
print("Vista previa de la muestra 0:\n", dataset[0]["text"][:250])

### Paso 5: Entrenamiento Acelerado con `SFTTrainer` y Optimizaciones de Unsloth

Entrenamos con la integracion nativa de Unsloth y `trl.SFTTrainer`:


In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./unsloth_gemma_json_output",
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    num_train_epochs=5,
    warmup_steps=2,
    logging_steps=1,
    optim="adamw_8bit",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_args,
)

print("Iniciando entrenamiento ultra-rapido con Unsloth...")
trainer.train()

### Paso 6: Inferencia de Alta Velocidad con `FastLanguageModel.for_inference`

Activamos el modo de inferencia acelerado de Unsloth (hasta 2x mas rapido que Hugging Face estandar):


In [ ]:
FastLanguageModel.for_inference(model)

test_ticket = (
    "Analiza el siguiente ticket de soporte y extrae la informacion en formato JSON con las claves categoria, urgencia, sentimiento y accion_sugerida:\n"
    "Ayuda urgente! Olvide la clave maestra del bucket S3 y mis pipelines de datos se detuvieron. Necesito asistencia prioritaria ahora mismo!"
)

messages = [{"role": "user", "content": test_ticket}]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda" if torch.cuda.is_available() else "cpu")

outputs = model.generate(input_ids=inputs, max_new_tokens=128, temperature=0.1, do_sample=False)
res_text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

print("=== RESPUESTA JSON ESTRUCTURADA GENERADA POR EL MODELO ===\n")
print(res_text.strip())

### Paso 7: Exportacion Directa a Formato GGUF para Produccion (Ollama / vLLM)

Unsloth permite guardar el modelo entrenado directamente en formato binario cuantizado **GGUF** con metodo de cuantizacion `q4_k_m` (excelente balance entre tamano y fidelidad).

A diferencia de formatos como `safetensors` que solo guardan tensores puros, **GGUF codifica simultaneamente los tensores y los metadatos estandarizados** (arquitectura, vocabulario del tokenizador, hiperparametros y plantillas de chat) en un unico archivo optimizado para carga rapida por memoria mapeada (`mmap`), adoptado nativamente por el ecosistema de Hugging Face y motores de inferencia como Ollama, llama.cpp y vLLM.

> **Referencia Oficial de Hugging Face:** Consulte la documentacion detallada sobre la arquitectura y tipos de cuantizacion de GGUF en [Hugging Face Hub GGUF Guide](https://huggingface.co/docs/hub/gguf).

In [ ]:
print("Exportando modelo a formato binario GGUF (q4_k_m)...")
try:
    model.save_pretrained_gguf("model_gguf", tokenizer, quantization_method="q4_k_m")
    print("Archivo GGUF guardado exitosamente en el directorio: model_gguf/")
except Exception as e:
    print(f"Nota sobre entorno: La compilacion de GGUF requiere dependencias de llama.cpp ({e}).")
    print("El comando oficial de exportacion es: model.save_pretrained_gguf('model_gguf', tokenizer, 'q4_k_m')")

### Paso 8: Creacion Automatizada de `Modelfile` para Servir con Ollama

Generamos el archivo de manifiesto `Modelfile` que permite importar y desplegar el modelo en **Ollama** con una sola linea de comando (sirviendo de conexion directa con la Sesion 4 del workshop):


In [ ]:
modelfile_lines = [
    "FROM ./model_gguf/unsloth.Q4_K_M.gguf",
    "",
    "PARAMETER temperature 0.2",
    "PARAMETER top_p 0.95",
    "PARAMETER stop \"<end_of_turn>\"",
    "",
    'TEMPLATE """{{ if .System }}<start_of_turn>system\n{{ .System }}<end_of_turn>\n{{ end }}{{ if .Prompt }}<start_of_turn>user\n{{ .Prompt }}<end_of_turn>\n{{ end }}<start_of_turn>model\n{{ .Response }}<end_of_turn>\n"""',
    "",
    'SYSTEM """Eres un clasificador de incidentes corporativos de TechCloud Pro. Respondes exclusivamente en formato JSON valido con claves: categoria, urgencia, sentimiento y accion_sugerida."""'
]
modelfile_content = "\n".join(modelfile_lines)

with open("Modelfile", "w") as f:
    f.write(modelfile_content)

print("Modelfile para Ollama generado exitosamente:")
print("--------------------------------------------------")
print(modelfile_content)
print("--------------------------------------------------")
print("\nComando para desplegar en tu servidor local de Ollama (Sesion 4):")
print("ollama create techcloud-classifier -f Modelfile")
print("ollama run techcloud-classifier")

### Paso 9: Limpieza de Memoria y Recursos

Liberacion de memoria VRAM:


In [ ]:
del trainer, model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Recursos liberados exitosamente.")